# Function Calling

Step 1: Original question<br>
messages = [<br>
    {"role": "user", <br>
     "content": "I just discovered the course. Can I join it?"}<br>
]

This is the memory so far. <br>

Step 2: Ask the model what to do <br>
response = openai_client.responses.create( <br>
    model="gpt-5.4-mini",<br>
    input=messages,<br>
    tools=[search_tool],<br>
)

The model returns a tool call:<br>
response.output[0]

Meaning:

“Call search with this query.”

Step 3: Run the real Python function <br>
call = response.output[0]<br>
args = json.loads(call.arguments)<br>

results = search(**args)

This executes the search and returns relevant FAQ documents.

Step 4: Add everything to memory

Conceptually, you add:<br>
user question <br>
model tool call<br>
tool/search result<br>
to messages.

The tool result is the FAQ context the model needs.

Step 5: Ask the model again<br>
response = openai_client.responses.create(<br>
    model="gpt-5.4-mini",<br>
    input=messages,<br>
    tools=[search_tool],<br>
)

Now the model sees:<br>
1. Original question <br>
2. Its decision to search <br>
3. Search results from the FAQ <br>
So it can answer. <br>

What does it use to answer? <br>
Mainly: <br>
original question + search results

The modified query was just an intermediate step to retrieve better documents.

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
# .strip() removes extra spaces and new lines from the beginning and end of a string.
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

# Asking without tools

In [4]:
# Asking without tools
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

# openai_client.responses.create() is a method call on the OpenAI client object.
# openai_client = object
# responses = part/service of the object
# create() = method that sends request to OpenAI and creates a response
# means: Send these messages to the model and get an AI response back.
response = openai_client.responses.create( 
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes — if the course is still open and you meet any prerequisites, you can usually join it.\n\nIf you want, I can help you with one of these:\n- check whether enrollment is still open,\n- figure out if you’re eligible,\n- or draft a message to the course organizer asking to join.\n\nIf you share the course name or a link, I can help you more specifically.'

The model answers from its general knowledge, something like "it depends on the course" or "check the course website". It doesn't know about our FAQ, so the answer is vague and not helpful. 

# Defining the tool

In [5]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

search_tool is a dictionary, but it follows a required schema format so the LLM knows how to call your Python function.

In [6]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [7]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? discovered the course join late enrollment"}', call_id='call_fbh5MF2hZMW25oqSssCjzL8m', name='search', type='function_call', id='fc_099d21558e9ce88a006a3eca1efb248192af48b9f813ee8f59', namespace=None, status='completed')]

In [70]:
response.output[0]

ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment eligibility can I join after start"}', call_id='call_vDJTJm9Ma5fmn9sw21UwsJxe', name='search', type='function_call', id='fc_052f2157765e9fcc006a3ee12b48c48191b7229c8094e2acec', namespace=None, status='completed')

In [8]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

## What is the point of search_tool?

It tells the LLM:

“There is a tool/function available called search. You can call it, and it needs one argument: query.”

search_tool is a dictionary, but it follows a required schema format so the LLM knows how to call your Python function.

In [9]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

The description is the most important field, because the model reads it to decide when to call the function. parameters is a JSON schema for the arguments, and we mark query as required so the model always fills it in.

# Sending the question with the tool


Now we send the same question as before, but this time we include the tool in the request:

In [55]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment eligibility can I join after start"}', call_id='call_vDJTJm9Ma5fmn9sw21UwsJxe', name='search', type='function_call', id='fc_052f2157765e9fcc006a3ee12b48c48191b7229c8094e2acec', namespace=None, status='completed')]

In [18]:
response

Response(id='resp_0ce903d83af99092006a3ed260dc80819e99fa9106f69cdb45', created_at=1782501984.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment late join available new student"}', call_id='call_PB7FYoOc1zBDrN6bugHDIBAC', name='search', type='function_call', id='fc_0ce903d83af99092006a3ed2617a64819e808db4e8fb4bed76', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='search', parameters={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query text to look up in the course FAQ.'}}, 'required': ['query'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Search the FAQ database for entries matching the given query.')], top_p=0.98, background=False, comple

output is an attribute/property of the response object.

response.output

Means:

“Get the value stored in output.”

It is not a method because there are no parentheses.

Why not output()? <br>
Because output is data, not an action.

In [40]:
# dir(response) shows you the available attributes and methods inside the response object.
dir(response)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_extra_info__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_on_complete__',
 '__pydantic_parent_namespace__',
 '__pydantic_post_init__',
 '__pydantic_private__',
 '__pydantic_root_model__

In [ ]:
for x in dir(response):
    if not x.startswith("_"):
       print(x)


background
completed_at
construct
conversation
copy
created_at
dict
error
from_orm
id
incomplete_details
instructions
json
max_output_tokens
max_tool_calls
metadata
model
model_computed_fields
model_config
model_construct
model_copy
model_dump
model_dump_json
model_extra
model_fields
model_fields_set
model_json_schema
model_parametrized_name
model_post_init
model_rebuild
model_validate
model_validate_json
model_validate_strings
moderation
object
output
output_text
parallel_tool_calls
parse_file
parse_obj
parse_raw
previous_response_id
prompt
prompt_cache_key
prompt_cache_retention
reasoning
safety_identifier
schema
schema_json
service_tier
status
temperature
text
to_dict
to_json
tool_choice
tools
top_logprobs
top_p
truncation
update_forward_refs
usage
user
validate


**The list comprehension version** <br>
What do I want to put in the new list? <br>

[VALUE_TO_KEEP for ITEM in LIST if CONDITION]

In [54]:
len(dir(response))

140

In [52]:
[x for x in dir(response) if not x.startswith("_")]

['background',
 'completed_at',
 'construct',
 'conversation',
 'copy',
 'created_at',
 'dict',
 'error',
 'from_orm',
 'id',
 'incomplete_details',
 'instructions',
 'json',
 'max_output_tokens',
 'max_tool_calls',
 'metadata',
 'model',
 'model_computed_fields',
 'model_config',
 'model_construct',
 'model_copy',
 'model_dump',
 'model_dump_json',
 'model_extra',
 'model_fields',
 'model_fields_set',
 'model_json_schema',
 'model_parametrized_name',
 'model_post_init',
 'model_rebuild',
 'model_validate',
 'model_validate_json',
 'model_validate_strings',
 'moderation',
 'object',
 'output',
 'output_text',
 'parallel_tool_calls',
 'parse_file',
 'parse_obj',
 'parse_raw',
 'previous_response_id',
 'prompt',
 'prompt_cache_key',
 'prompt_cache_retention',
 'reasoning',
 'safety_identifier',
 'schema',
 'schema_json',
 'service_tier',
 'status',
 'temperature',
 'text',
 'to_dict',
 'to_json',
 'tool_choice',
 'tools',
 'top_logprobs',
 'top_p',
 'truncation',
 'update_forward_refs',


In [28]:
response.model_dump().keys() 

dict_keys(['id', 'created_at', 'error', 'incomplete_details', 'instructions', 'metadata', 'model', 'object', 'output', 'parallel_tool_calls', 'temperature', 'tool_choice', 'tools', 'top_p', 'background', 'completed_at', 'conversation', 'max_output_tokens', 'max_tool_calls', 'moderation', 'previous_response_id', 'prompt', 'prompt_cache_key', 'prompt_cache_retention', 'reasoning', 'safety_identifier', 'service_tier', 'status', 'text', 'top_logprobs', 'truncation', 'usage', 'user', 'billing', 'frequency_penalty', 'presence_penalty', 'store', 'tool_usage'])

In [29]:
response.model_dump().keys() == 'output'

False

What is "in"?

Here, in is the membership operator. <br>
It checks if something exists inside a collection.

In [38]:
"output" in response.model_dump().keys()

True

In [56]:
# model_dump() is a method that returns a dictionary representation of the response object.
response.model_dump()

{'id': 'resp_052f2157765e9fcc006a3ee12a9ea4819186e429919f3077b5',
 'created_at': 1782505770.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'gpt-5.4-mini-2026-03-17',
 'object': 'response',
 'output': [{'arguments': '{"query":"join course discovered late enrollment eligibility can I join after start"}',
   'call_id': 'call_vDJTJm9Ma5fmn9sw21UwsJxe',
   'name': 'search',
   'type': 'function_call',
   'id': 'fc_052f2157765e9fcc006a3ee12b48c48191b7229c8094e2acec',
   'namespace': None,
   'status': 'completed'}],
 'parallel_tool_calls': True,
 'temperature': 1.0,
 'tool_choice': 'auto',
 'tools': [{'name': 'search',
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'Search query text to look up in the course FAQ.'}},
    'required': ['query'],
    'additionalProperties': False},
   'strict': True,
   'type': 'function',
   'defer_loading': None,
   'description': 'Search the FAQ databa

In [39]:
# ["output"] Accesses the value stored under the key "output".
response.model_dump()["output"]

[{'arguments': '{"query":"Can I join the course if I just discovered it? enrollment late join available new student"}',
  'call_id': 'call_PB7FYoOc1zBDrN6bugHDIBAC',
  'name': 'search',
  'type': 'function_call',
  'id': 'fc_0ce903d83af99092006a3ed2617a64819e808db4e8fb4bed76',
  'namespace': None,
  'status': 'completed'}]

In [33]:
response.model_dump().get("output")

[{'arguments': '{"query":"Can I join the course if I just discovered it? enrollment late join available new student"}',
  'call_id': 'call_PB7FYoOc1zBDrN6bugHDIBAC',
  'name': 'search',
  'type': 'function_call',
  'id': 'fc_0ce903d83af99092006a3ed2617a64819e808db4e8fb4bed76',
  'namespace': None,
  'status': 'completed'}]

# Executing the function and sending the result back


Big picture

The model decided:

“I need to call the search tool.”

But the model does not actually run your Python function.

It only returns a tool call like:

search({"query": "Can I join the course?"})

Then your code does 3 things:

1. Read the tool call from the model response
2. Convert the JSON arguments into Python
3. Run the real Python search function
4. Convert the results back to JSON so we can send them back to the mode

In [ ]:
# Loads Python’s json module.
import json

# response.output is a list.
# response.output[0] is the tool call the model requested.
call = response.output[0]
# call.arguments is a JSON string.
# json.loads() converts it into a Python dictionary
args = json.loads(call.arguments)

# This calls our real Python function
results = search(**args)
# json.dumps() converts it into a JSON string.
# indent=2 makes it pretty/readable.
result_json = json.dumps(results, indent=2)

In [60]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment eligibility can I join after start"}', call_id='call_vDJTJm9Ma5fmn9sw21UwsJxe', name='search', type='function_call', id='fc_052f2157765e9fcc006a3ee12b48c48191b7229c8094e2acec', namespace=None, status='completed')

In [63]:
call.arguments

'{"query":"join course discovered late enrollment eligibility can I join after start"}'

In [62]:
args

{'query': 'join course discovered late enrollment eligibility can I join after start'}

When do we use serch(**args) ? <br>

Use ** when you have a dictionary and want to pass it as keyword arguments.

cause **args becomes: <br>
search(query="Can I join the course?")

Now we send this result back to the model. First, we add the model's output to the conversation history - the model needs to see its own function call. Then we add the tool result

In [64]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [69]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch

# Asking the model again
We call the API a second time with the expanded history:



In [71]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join and start learning.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.'